# Tutorial 13: Configuring LAILA from a TOML file (or terminal)

Every CLI-capable LAILA class resolves its field values in this order:

1. Explicit kwarg passed at construction time.
2. The matching subtree under `laila.args`.
3. The class's declared default.

`laila.read_args(source)` populates `laila.args` from a TOML / JSON / `.env` / XML file, or from terminal arguments.

You will:

- Load a TOML file via `laila.read_args`
- Inspect the resulting `laila.args` tree
- Construct a CLI-capable class and watch defaults flow in from `laila.args`
- Load `.env` and terminal-style arguments the same way

**Prerequisites:** `pip install laila-core`.

In [ ]:
import tempfile
from pathlib import Path

import laila

workdir = Path(tempfile.mkdtemp(prefix="laila_args_"))
print("workdir:", workdir)

## TOML — the most common form

Write a `config.toml` with top-level keys and nested `[table]` blocks. Nested tables map onto nested DotMap attributes on `laila.args`.

In [ ]:
toml_path = workdir / "config.toml"
toml_path.write_text("""
AWS_REGION = "us-east-1"
AWS_BUCKET_NAME = "my-bucket"

[secrets]
api_token = "sk-demo-1234"
""")

laila.read_args(str(toml_path))

print("AWS_REGION:", laila.args.AWS_REGION)
print("bucket:    ", laila.args.AWS_BUCKET_NAME)
print("api token: ", laila.args.secrets.api_token)

## `.env` files

`.env` files are parsed line-by-line as `KEY=value`. Use them for secrets you want kept out of source control.

In [ ]:
env_path = workdir / "secrets.env"
env_path.write_text("AWS_ACCESS_KEY_ID=AKIAEXAMPLE\nAWS_SECRET_ACCESS_KEY=very-secret\n")
laila.read_args(str(env_path))

print("access key:", laila.args.AWS_ACCESS_KEY_ID)
print("secret key:", laila.args.AWS_SECRET_ACCESS_KEY[:8], "...")

## Terminal-style arguments

`laila.read_args("terminal", terminal_args=[...])` parses `key=value` pairs from the list. This is the form `from_terminal` understands — it is **not** GNU-style `--flag value`, but the literal `--flag=value` (or just `flag=value`) is fine.

In [ ]:
laila.read_args("terminal", terminal_args=["LOG_LEVEL=DEBUG", "PROJECT=demo"])
print("log level:", laila.args.LOG_LEVEL)
print("project:  ", laila.args.PROJECT)

## CLI-capable classes pick up defaults from `laila.args`

Any class derived from `_LAILA_CLI_CAPABLE_CLASS` walks `laila.args` during validation. If a field is **not** passed explicitly and a matching key is found, the value is used as the default. For example, an `S3Pool` constructed without arguments would pick up `laila.args.AWS_REGION` and `laila.args.AWS_BUCKET_NAME` from the keys you loaded above.

For now, just confirm `laila.args` holds the merged values:

In [ ]:
print("laila.args top-level keys:", list(laila.args.toDict().keys()))
print("AWS_REGION value:        ", laila.args.AWS_REGION)
print("secrets subtree:         ", laila.args.secrets.toDict())

## Round-tripping the entire environment

`laila.args.environment` is special. Assigning a payload with non-empty `policies` triggers a full `laila.terminate(...)` and rebuild of every described policy. This is what tutorials 8a and 8b use to capture and restore a setup across processes.

In [ ]:
snapshot_keys = list(laila.args.toDict().keys())
print("currently in laila.args:", snapshot_keys)

## Gotchas

- `from_terminal` parses **`key=value`** tokens. A bare `--flag` followed by a separate value will not work.
- Deeply nested TOML tables work, but `ArgReader._apply` flattens with `_` at the first level — check `laila.args.toDict()` if a lookup feels off.
- `read_args` **merges** into `laila.args`. To start fresh, restart the kernel or call `laila.terminate()` first.

## Summary

- `laila.read_args(path)` supports `.toml`, `.json`, `.env`, `.xml`.
- `laila.read_args("terminal", terminal_args=[...])` accepts `key=value` tokens.
- Subsequent CLI-capable class construction picks up the loaded values as defaults.
- Assigning `laila.args.environment = {...}` triggers a full process-wide reload.